In [ ]:
## Goal

The goal of this notebook is to build the datasets used in this project:

- FIFA rankings dataset (from API)
- World Cup titles dataset (manually created)
- Historical match results (from Kaggle)

These datasets are later used for analysis and simulation.

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("FIFA_API")

headers = {
    'Authorization': f'Bearer {api_key}'
}




In [ ]:
## FIFA Rankings Data (API)

We use the Zyla Labs FIFA API to retrieve the most recent FIFA rankings.
This includes team name, rank, and points.

Steps:
1. Request available ranking dates
2. Select the most recent dataset
3. Extract relevant fields (rank, team name, points)

In [2]:
import requests

# The endpoint URL for Ranking by Date
url = "https://zylalabs.com/api/3028/fifa+ranking+api/3199/historical+ranking"

# Replace 'YOUR_API_KEY' with your actual Zyla Labs access key
headers = {
    'Authorization': f'{api_key}'
}


try:
    response = requests.get(url, headers=headers)
    
    # Check if the request was successful
    if response.status_code == 200:
        data = response.json()
        print(data)
        
        # Loop through the first few rankings
    else:
        print(f"Error: {response.status_code} - {response.text}")

except Exception as e:
    print(f"An error occurred: {e}")

[{'id': 'FRS_Male_Football_20260119', 'date': '01 April 2026'}, {'id': 'FRS_Male_Football_20251219', 'date': '19 January 2026'}, {'id': 'FRS_Male_Football_20251119', 'date': '23 December 2025'}, {'id': 'FRS_Male_Football_20251015', 'date': '19 November 2025'}, {'id': 'FRS_Male_Football_20250910', 'date': '17 October 2025'}, {'id': 'id14870', 'date': '18 September 2025'}, {'id': 'id14800', 'date': '10 July 2025'}, {'id': 'id14702', 'date': '03 April 2025'}, {'id': 'id14597', 'date': '19 December 2024'}, {'id': 'id14576', 'date': '28 November 2024'}, {'id': 'id14541', 'date': '24 October 2024'}, {'id': 'id14506', 'date': '19 September 2024'}, {'id': 'id14443', 'date': '18 July 2024'}, {'id': 'id14415', 'date': '20 June 2024'}, {'id': 'id14338', 'date': '04 April 2024'}, {'id': 'id14289', 'date': '15 February 2024'}, {'id': 'id14233', 'date': '21 December 2023'}, {'id': 'id14212', 'date': '30 November 2023'}, {'id': 'id14177', 'date': '26 October 2023'}, {'id': 'id14142', 'date': '21 Sept

In [3]:
import requests

url = "https://zylalabs.com/api/3028/fifa+ranking+api/3200/ranking+by+date"

# Use the full string including the numbers and the pipe symbol
headers = {
    'Authorization': f'{api_key}'
}

params = {
    'id': 'id14870'
}

response = requests.get(url, headers=headers, params=params)

if response.status_code == 200:
    data = response.json()
    print(data)
else:
    print(f"Failed: {response.status_code}")

{'date': '18 September, 2025', 'ranking': [{'rank': 1, 'flag': 'https://api.fifa.com/api/v3/picture/flags-sq-2/ESP', 'name': 'Spain', 'previous_rank': 2, 'points': 1875.37, 'previous_points': 1867.09, 'confederation': 'UEFA'}, {'rank': 2, 'flag': 'https://api.fifa.com/api/v3/picture/flags-sq-2/FRA', 'name': 'France', 'previous_rank': 3, 'points': 1870.92, 'previous_points': 1862.03, 'confederation': 'UEFA'}, {'rank': 3, 'flag': 'https://api.fifa.com/api/v3/picture/flags-sq-2/ARG', 'name': 'Argentina', 'previous_rank': 1, 'points': 1870.32, 'previous_points': 1885.36, 'confederation': 'CONMEBOL'}, {'rank': 4, 'flag': 'https://api.fifa.com/api/v3/picture/flags-sq-2/ENG', 'name': 'England', 'previous_rank': 4, 'points': 1820.44, 'previous_points': 1813.32, 'confederation': 'UEFA'}, {'rank': 5, 'flag': 'https://api.fifa.com/api/v3/picture/flags-sq-2/POR', 'name': 'Portugal', 'previous_rank': 6, 'points': 1779.55, 'previous_points': 1770.53, 'confederation': 'UEFA'}, {'rank': 6, 'flag': 'ht

In [4]:
raw_list = data.get('ranking',[])
print(raw_list)
df = pd.DataFrame(raw_list)


[{'rank': 1, 'flag': 'https://api.fifa.com/api/v3/picture/flags-sq-2/ESP', 'name': 'Spain', 'previous_rank': 2, 'points': 1875.37, 'previous_points': 1867.09, 'confederation': 'UEFA'}, {'rank': 2, 'flag': 'https://api.fifa.com/api/v3/picture/flags-sq-2/FRA', 'name': 'France', 'previous_rank': 3, 'points': 1870.92, 'previous_points': 1862.03, 'confederation': 'UEFA'}, {'rank': 3, 'flag': 'https://api.fifa.com/api/v3/picture/flags-sq-2/ARG', 'name': 'Argentina', 'previous_rank': 1, 'points': 1870.32, 'previous_points': 1885.36, 'confederation': 'CONMEBOL'}, {'rank': 4, 'flag': 'https://api.fifa.com/api/v3/picture/flags-sq-2/ENG', 'name': 'England', 'previous_rank': 4, 'points': 1820.44, 'previous_points': 1813.32, 'confederation': 'UEFA'}, {'rank': 5, 'flag': 'https://api.fifa.com/api/v3/picture/flags-sq-2/POR', 'name': 'Portugal', 'previous_rank': 6, 'points': 1779.55, 'previous_points': 1770.53, 'confederation': 'UEFA'}, {'rank': 6, 'flag': 'https://api.fifa.com/api/v3/picture/flags-sq

In [ ]:
## Data Cleaning and Export

We clean the dataset and keep only the necessary columns.
Then we export the dataset to a CSV file to be used in the package.

In [5]:
cols_to_keep = ['rank', 'name', 'points']
df_filtered = df[cols_to_keep]
print(df_filtered)

      rank                    name   points
0      1.0                   Spain  1875.37
1      2.0                  France  1870.92
2      3.0               Argentina  1870.32
3      4.0                 England  1820.44
4      5.0                Portugal  1779.55
..     ...                     ...      ...
206  207.0       US Virgin Islands   779.71
207  208.0  British Virgin Islands   768.50
208  209.0                Anguilla   758.52
209  210.0              San Marino   733.23
210    NaN                 Eritrea   855.56

[211 rows x 3 columns]


In [6]:
df_filtered['rank'] = pd.to_numeric(df_filtered['rank'])
df_filtered['points'] = pd.to_numeric(df_filtered['points'])
df_filtered["rank"] = df_filtered["rank"].astype("Int64")

print(df_filtered.head())

   rank       name   points
0     1      Spain  1875.37
1     2     France  1870.92
2     3  Argentina  1870.32
3     4    England  1820.44
4     5   Portugal  1779.55


In [ ]:
## Data Cleaning and Export

We clean the dataset and keep only the necessary columns.
Then we export the dataset to a CSV file to be used in the package.

In [7]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = file_path = "results.csv"

# Load the latest version
results_df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "martj42/international-football-results-from-1872-to-2017",
  file_path,
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", results_df.head())

/Users/santiago/final_stats_386/final_project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/_y/ns92xnlj51d2sj00hr93x7r80000gn/T/ipykernel_8057/3667409590.py:10: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  results_df = kagglehub.load_dataset(


First 5 records:          date home_team away_team  home_score  away_score tournament     city  \
0  1872-11-30  Scotland   England         0.0         0.0   Friendly  Glasgow   
1  1873-03-08   England  Scotland         4.0         2.0   Friendly   London   
2  1874-03-07  Scotland   England         2.0         1.0   Friendly  Glasgow   
3  1875-03-06   England  Scotland         2.0         2.0   Friendly   London   
4  1876-03-04  Scotland   England         3.0         0.0   Friendly  Glasgow   

    country  neutral  
0  Scotland    False  
1   England    False  
2  Scotland    False  
3   England    False  
4  Scotland    False  


In [8]:
df_filtered.to_csv('data/rankings.csv', index=False)


In [9]:
%pip install -e .

/Users/santiago/final_stats_386/final_project/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
from wc_sim import calculate_weighted_rating
import pandas as pd

ranks  = pd.read_csv('data/rankings.csv')
titles = pd.read_csv('data/titles.csv')
df_weighted = calculate_weighted_rating(ranks, titles)

In [11]:
print(df_weighted.head(10))

   rank         name   points  wc_titles  weighted_rating
0     1    Argentina  1870.32          3          2020.32
1     2       Brazil  1761.60          5          2011.60
2     3       France  1870.92          2          1970.92
3     4        Spain  1875.37          1          1925.37
4     5        Italy  1710.06          4          1910.06
5     6      Germany  1704.27          4          1904.27
6     7      England  1820.44          1          1870.44
7     8     Portugal  1779.55          0          1779.55
8     9      Uruguay  1673.65          2          1773.65
9    10  Netherlands  1754.17          0          1754.17


In [12]:
from wc_sim import simulate_match

winner = simulate_match("Brazil", "Argentina", df_weighted)
print(winner)

Argentina


In [13]:
from wc_sim import simulate_match_n

simulate_match_n("Brazil", "Argentina", df_weighted, n=1000)

,team,wins,win_pct
0,Brazil,482,48.2
1,Argentina,518,51.8


In [14]:
from wc_sim import validate_weighted_rating

validate_weighted_rating(results_df, df_weighted)


,metric,correct_predictions,total_matches,accuracy_pct
0,points,20701,29668,69.78
1,weighted_rating,20795,29668,70.09
